# Create Founder Level Event Dataset
This notebook processes the raw position history dataset and compiles a founder-level event dataset focusing on the founding events (when individuals became entrepreneurs).

## Methodology and Key Decisions
1. **Sharding for Scalability**: The raw positions CSV (`Founder_Full_Position_List_[US-2000-2023].csv`) is 5.8 GB. To process it efficiently without out-of-memory errors on typical development environments, we sharded the positions into 20 smaller files partitioned by `user_id % 20`. This ensures that all records for a given user are kept together in the same shard.
2. **Chronological Reordering**: For each user, positions are sorted chronologically by `startdate` (falling back to `startyear`) to reconstruct their complete work history.
3. **Founding Event Identification**: We load the mapping file `Founder_Position_List_[US-2000-2023].csv` into memory as a set of founding `position_id`s, enabling fast $O(1)$ lookups.
4. **Pre-Founding Controls**: For each founding event:
   * **Years to Founding**: Calculates the duration (in years) from their very first position to this founding event.
   * **Last Prior Salary**: Extracts the salary of the last non-founding position held immediately prior to this venture.
   * **Last Firm Prior**: Keeps the firm ID (`rcid` or `company_linkedin_url`) of the immediately preceding position.
   * **Prior Seniority Counts**: Tallies the total count of positions held prior to this event, the maximum seniority level achieved, and the counts of prior positions at each seniority level (1-7).
5. **Post-Founding Outcomes**:
   * Checks if the venture is ongoing (no end date) or has ended.
   * Calculates the overall duration of the venture in years.


In [1]:
import pandas as pd
import numpy as np
import os
import re
import time
import csv

base_dir = ".."
positions_path = os.path.join(base_dir, "D - Data/D1 - Extracted Datasets/Founder_Full_Position_List_[US-2000-2023].csv")
mapping_path = os.path.join(base_dir, "D - Data/D1 - Extracted Datasets/Founder_Position_List_[US-2000-2023].csv")
shard_dir = os.path.join(base_dir, "D - Data/D1 - Extracted Datasets/temp_shards")
out_dir = os.path.join(base_dir, "D - Data/D2 - Datasets for Matching")

os.makedirs(out_dir, exist_ok=True)

# 1. Load mapping set of founding positions
print("Loading founding position IDs...")
df_map = pd.read_csv(mapping_path, usecols=['position_id'], low_memory=False)
founding_positions = set(df_map['position_id'].values)
print(f"Loaded {len(founding_positions):,} founding position IDs.")

# Helper to parse dates
def parse_date_vectorized(df, date_col, year_col):
    dt_col = pd.to_datetime(df[date_col].astype(str).str.strip(), errors='coerce')
    year_str = df[year_col].astype(float).fillna(np.nan).apply(lambda y: f"{int(y)}-01-01" if pd.notnull(y) and 1950 <= y <= 2026 else np.nan)
    year_dt = pd.to_datetime(year_str, errors='coerce')
    return dt_col.fillna(year_dt)

def process_shard_for_events(shard_path, founding_positions):
    df_shard = pd.read_csv(shard_path, low_memory=False)
    
    # Parse dates (vectorized)
    df_shard['parsed_start'] = parse_date_vectorized(df_shard, 'startdate', 'startyear')
    df_shard['parsed_end'] = parse_date_vectorized(df_shard, 'enddate', 'endyear')
    
    # Sort chronologically per user
    df_shard = df_shard.sort_values(by=['user_id', 'parsed_start', 'parsed_end']).copy()
    
    # Found_Pos indicator
    df_shard['is_founding'] = df_shard['position_id'].apply(lambda pid: 1 if pid in founding_positions else 0)
    
    # Compute first job date per user
    df_shard['first_job_date'] = df_shard.groupby('user_id')['parsed_start'].transform('min')
    
    # Compute founding event number
    df_shard['founding_event_number'] = df_shard.groupby('user_id')['is_founding'].cumsum()
    
    # Columns to ffill from last pre-founding position
    cols_to_extract = [
        'position_id', 'is_founding', 'title_raw', 'job_category_v2', 'seniority', 'salary', 'naics_description',
        'role_k50_v3', 'role_k150_v3', 'role_k500_v3', 'role_k1000_v3', 'role_k1500_v3', 'role_k5000_v3', 'role_k10000_v3', 'role_k15000_v3',
        'metro_area', 'msa', 'country', 'rcid', 'rics_k50', 'rics_k200', 'rics_k400', 'onet_code', 'onet_title'
    ]
    
    # Group and ffill
    for col in cols_to_extract:
        temp = df_shard[col].where(df_shard['is_founding'] == 0)
        df_shard[f'last_pre_{col}'] = temp.groupby(df_shard['user_id']).ffill()
        
    # Prior non-founding positions count
    prior_nf = (df_shard['is_founding'] == 0).astype(int)
    cum_nf = prior_nf.groupby(df_shard['user_id']).cumsum()
    df_shard['total_prior_positions'] = cum_nf - prior_nf

    # Seniority counts
    for k in range(1, 8):
        is_sen_k = ((df_shard['seniority'].astype(float).fillna(0).astype(int) == k) & (df_shard['is_founding'] == 0)).astype(int)
        cum_sen_k = is_sen_k.groupby(df_shard['user_id']).cumsum()
        df_shard[f'prior_sen_pos_count_{k}'] = cum_sen_k - is_sen_k
        
    # Max seniority
    sen_val = df_shard['seniority'].astype(float).where(df_shard['is_founding'] == 0).fillna(0)
    running_max_sen = sen_val.groupby(df_shard['user_id']).cummax()
    df_shard['max_prior_seniority'] = running_max_sen.groupby(df_shard['user_id']).shift(1).fillna(0).astype(int)
    
    # Ongoing status and duration
    df_shard['is_ongoing'] = df_shard['parsed_end'].isnull().astype(int)
    df_shard['has_end_date'] = (df_shard['is_ongoing'] == 0).astype(int)
    
    now_ts = pd.Timestamp.now()
    end_date_for_duration = df_shard['parsed_end'].fillna(now_ts)
    df_shard['venture_duration_years'] = (end_date_for_duration - df_shard['parsed_start']).dt.days / 365.25
    
    # Filter to founding rows only
    df_events_shard = df_shard[df_shard['is_founding'] == 1].copy()
    
    # Process event rows
    df_events_shard['years_since_first_job'] = (df_events_shard['parsed_start'] - df_events_shard['first_job_date']).dt.days / 365.25
    df_events_shard['last_pre_founding_salary'] = df_events_shard['last_pre_salary']
    df_events_shard['last_firm_id'] = df_events_shard['last_pre_rcid']
    df_events_shard['venture_position_id'] = df_events_shard['position_id']
    df_events_shard['venture_firm_id'] = df_events_shard['rcid']
    
    df_events_shard['venture_metro_area'] = df_events_shard['metro_area']
    df_events_shard['venture_msa'] = df_events_shard['msa']
    df_events_shard['venture_country'] = df_events_shard['country']
    df_events_shard['venture_rics_k50'] = df_events_shard['rics_k50']
    df_events_shard['venture_rics_k200'] = df_events_shard['rics_k200']
    df_events_shard['venture_rics_k400'] = df_events_shard['rics_k400']
    
    df_events_shard['venture_role_k50_v3'] = df_events_shard['role_k50_v3']
    df_events_shard['venture_role_k150_v3'] = df_events_shard['role_k150_v3']
    df_events_shard['venture_role_k500_v3'] = df_events_shard['role_k500_v3']
    df_events_shard['venture_role_k1000_v3'] = df_events_shard['role_k1000_v3']
    df_events_shard['venture_role_k1500_v3'] = df_events_shard['role_k1500_v3']
    df_events_shard['venture_role_k5000_v3'] = df_events_shard['role_k5000_v3']
    df_events_shard['venture_role_k10000_v3'] = df_events_shard['role_k10000_v3']
    df_events_shard['venture_role_k15000_v3'] = df_events_shard['role_k15000_v3']
    
    event_cols = [
        'user_id', 'founding_event_number', 'years_since_first_job', 'last_pre_founding_salary', 'last_firm_id',
        'venture_position_id', 'venture_firm_id', 'venture_metro_area', 'venture_msa', 'venture_country',
        'venture_rics_k50', 'venture_rics_k200', 'venture_rics_k400',
        'venture_role_k50_v3', 'venture_role_k150_v3', 'venture_role_k500_v3', 'venture_role_k1000_v3',
        'venture_role_k150_v3', 'venture_role_k1500_v3', 'venture_role_k5000_v3', 'venture_role_k10000_v3', 'venture_role_k15000_v3', # Correct columns
        'is_ongoing', 'has_end_date', 'venture_duration_years', 'total_prior_positions', 'max_prior_seniority'
    ] + [f'prior_sen_pos_count_{k}' for k in range(1, 8)]
    
    # Ensure unique columns list
    event_cols_unique = []
    for col in event_cols:
        if col not in event_cols_unique:
            event_cols_unique.append(col)
            
    df_events_out = df_events_shard[event_cols_unique]
    
    # Process details rows
    df_events_shard['prior_position_id'] = df_events_shard['last_pre_position_id']
    df_events_shard['prior_is_founder'] = df_events_shard['last_pre_is_founding'].fillna(0).astype(int)
    df_events_shard['prior_title_raw'] = df_events_shard['last_pre_title_raw']
    df_events_shard['prior_job_category_v2'] = df_events_shard['last_pre_job_category_v2']
    df_events_shard['prior_seniority'] = df_events_shard['last_pre_seniority']
    df_events_shard['prior_salary'] = df_events_shard['last_pre_salary']
    df_events_shard['prior_naics_description'] = df_events_shard['last_pre_naics_description']
    
    df_events_shard['prior_role_k50_v3'] = df_events_shard['last_pre_role_k50_v3']
    df_events_shard['prior_role_k150_v3'] = df_events_shard['last_pre_role_k150_v3']
    df_events_shard['prior_role_k500_v3'] = df_events_shard['last_pre_role_k500_v3']
    df_events_shard['prior_role_k1000_v3'] = df_events_shard['last_pre_role_k1000_v3']
    df_events_shard['prior_role_k1500_v3'] = df_events_shard['last_pre_role_k1500_v3']
    df_events_shard['prior_role_k5000_v3'] = df_events_shard['last_pre_role_k5000_v3']
    df_events_shard['prior_role_k10000_v3'] = df_events_shard['last_pre_role_k10000_v3']
    df_events_shard['prior_role_k15000_v3'] = df_events_shard['last_pre_role_k15000_v3']
    
    df_events_shard['prior_metro_area'] = df_events_shard['last_pre_metro_area']
    df_events_shard['prior_msa'] = df_events_shard['last_pre_msa']
    df_events_shard['prior_country'] = df_events_shard['last_pre_country']
    
    df_events_shard['prior_rcid'] = df_events_shard['last_pre_rcid']
    df_events_shard['prior_rics_k50'] = df_events_shard['last_pre_rics_k50']
    df_events_shard['prior_rics_k200'] = df_events_shard['last_pre_rics_k200']
    df_events_shard['prior_rics_k400'] = df_events_shard['last_pre_rics_k400']
    df_events_shard['prior_onet_code'] = df_events_shard['last_pre_onet_code']
    df_events_shard['prior_onet_title'] = df_events_shard['last_pre_onet_title']
    
    detail_cols = [
        'user_id', 'founding_event_number', 'prior_position_id', 'prior_is_founder', 'prior_title_raw',
        'prior_job_category_v2', 'prior_seniority', 'prior_salary', 'prior_naics_description',
        'prior_role_k50_v3', 'prior_role_k150_v3', 'prior_role_k500_v3', 'prior_role_k1000_v3', 'prior_role_k1500_v3',
        'prior_role_k5000_v3', 'prior_role_k10000_v3', 'prior_role_k15000_v3',
        'prior_metro_area', 'prior_msa', 'prior_country',
        'prior_rcid', 'prior_rics_k50', 'prior_rics_k200', 'prior_rics_k400', 'prior_onet_code', 'prior_onet_title'
    ]
    df_details_out = df_events_shard[detail_cols]
    
    return df_events_out, df_details_out

# Process shards
all_events_dfs = []
all_details_dfs = []
start_time = time.time()

# Verify that sharding has run
if not os.path.exists(shard_dir) or len(os.listdir(shard_dir)) == 0:
    print("Shard directory empty or not found. Please run the sharding script first.")
else:
    shard_files = sorted([f for f in os.listdir(shard_dir) if f.startswith("shard_")])
    for idx, f in enumerate(shard_files):
        shard_path = os.path.join(shard_dir, f)
        print(f"Processing shard {idx+1}/{len(shard_files)} ({f})...")
        shard_events_df, shard_details_df = process_shard_for_events(shard_path, founding_positions)
        all_events_dfs.append(shard_events_df)
        all_details_dfs.append(shard_details_df)
        
    print(f"Processed all shards in {time.time() - start_time:.2f} seconds.")

# Save final events dataset
if all_events_dfs:
    df_events = pd.concat(all_events_dfs, ignore_index=True)
    out_path = os.path.join(out_dir, "Founder_Level_Event_Dataset.csv")
    df_events.to_csv(out_path, index=False)
    print(f"Saved aggregated event dataset to: {out_path}")
    print(f"Total rows: {len(df_events):,}")
    print(df_events.describe())

# Save final details dataset
if all_details_dfs:
    df_details = pd.concat(all_details_dfs, ignore_index=True)
    details_out_path = os.path.join(out_dir, "Founder_Prior_Position_Details.csv")
    df_details.to_csv(details_out_path, index=False)
    print(f"Saved prior position details dataset to: {details_out_path}")
    print(f"Total rows: {len(df_details):,}")


Loading founding position IDs...
Loaded 1,239,185 founding position IDs.
Processing shard 1/20 (shard_0.csv)...


Processing shard 2/20 (shard_1.csv)...


Processing shard 3/20 (shard_10.csv)...


Processing shard 4/20 (shard_11.csv)...


Processing shard 5/20 (shard_12.csv)...


...

Processing shard 20/20 (shard_9.csv)...


Processed all shards in 68.13 seconds.


Saved aggregated event dataset to: ../D - Data/D2 - Datasets for Matching/Founder_Level_Event_Dataset.csv
Total rows: 1,239,185


            user_id  founding_event_number  years_since_first_job  \
count  1.239185e+06           1.239185e+06           1.239185e+06   
mean   5.128306e+08           1.229310e+00           9.354806e+00   
std    4.859764e+08           6.150454e-01           8.482469e+00   
min    1.000086e+06           1.000000e+00           0.000000e+00   
25%    1.995276e+08           1.000000e+00           2.584531e+00   
50%    4.093790e+08           1.000000e+00           7.334702e+00   
75%    6.312193e+08           1.000000e+00           1.400137e+01   
max    2.219679e+09           2.700000e+01           7.091581e+01   

       last_pre_founding_salary  last_firm_id  venture_position_id  \
count              1.075067e+06  9.891310e+05         1.239185e+06   
mean               9.718917e+04  9.440370e+06         4.295926e+15   
std                7.442624e+04  2.198831e+07         5.327175e+18   
min                5.802260e+02  1.000000e+00        -9.223369e+18   
25%                4.327558e

Saved prior position details dataset to: ../D - Data/D2 - Datasets for Matching/Founder_Prior_Position_Details.csv
Total rows: 1,239,185
